# 03 - Feature Engineering

The project engineers two families of features:

| Module | Family | Highlights |
|---|---|---|
| `src/features/transaction_features.py` | transaction-level | `amount_log`, `is_large_transaction`, type one-hots, hour/day, **balance-consistency errors** |
| `src/features/behavioral_features.py` | per-customer, *as-of-now* | `customer_avg_amount`, `amount_zscore`, `transactions_last_24_steps`, unique recipients |

**Prerequisite:** `python main.py` (see `01_data_exploration.ipynb`).

In [ ]:
import sys
from pathlib import Path

# Make 'import src...' work no matter where the kernel was started from.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
FEATURED_PATH = PROJECT_ROOT / "data" / "processed" / "featured_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"

print("Project root:", PROJECT_ROOT)

## 1. Build the features live on a small sample

In [ ]:
from src.features.transaction_features import create_transaction_features
from src.features.behavioral_features import create_behavioral_features

base = pd.read_csv(DATA_PATH, nrows=5_000)
threshold = base["amount"].median()   # normally fitted on TRAIN rows only (see section 3)

tx = create_transaction_features(base, large_amount_threshold=threshold)
featured = create_behavioral_features(tx)

new_cols = sorted(set(featured.columns) - set(base.columns))
print(f"{len(new_cols)} new features:")
for c in new_cols:
    print("  -", c)

In [ ]:
# Invariants: the label and the row order must survive feature engineering.
assert featured["isfraud"].equals(base["isfraud"]), "labels were modified!"
assert featured["step"].equals(base["step"]), "row order changed!"
print("labels and row order preserved")

In [ ]:
cols = ["type", "amount", "is_transfer", "origin_balance_error",
        "destination_balance_inconsistent", "amount_to_origin_balance_ratio",
        "customer_transaction_count", "amount_to_customer_avg_ratio",
        "transactions_last_24_steps"]
featured[cols].head(8)

## 2. Balance-consistency features (the interesting part)

The simulator updates balances according to the transaction type:

- **TRANSFER / CASH_OUT / DEBIT** - origin debited, destination credited
- **CASH_IN** - origin *credited*, destination *debited*
- **PAYMENT** - origin debited, destination balance *never changes*

`create_transaction_features` computes the *expected* balance after the
transaction and stores `|expected - recorded|` as `origin_balance_error` /
`destination_balance_error`. Consistent rows get 0; a tampered or impossible
row gets a large value. The cell below builds 4 hand-crafted rows - 3 correct,
1 with a broken destination balance.

In [ ]:
edge = pd.DataFrame({
    "step": [1, 1, 1, 1],
    "type": ["TRANSFER", "CASH_IN", "PAYMENT", "TRANSFER"],
    "amount": [100.0, 100.0, 100.0, 100.0],
    "nameOrig": ["A", "B", "C", "D"],
    "oldbalanceOrg": [1000.0, 500.0, 300.0, 1000.0],
    "newbalanceOrig": [900.0, 600.0, 200.0, 900.0],
    "nameDest": ["X", "Y", "Z", "W"],
    "oldbalanceDest": [100.0, 200.0, 0.0, 100.0],
    # Last row is wrong on purpose: a TRANSFER of 100 must credit the
    # destination with +100 (100 -> 200), but only 50 arrived.
    "newbalanceDest": [200.0, 100.0, 0.0, 150.0],
    "isfraud": [0, 0, 0, 1],
    "isflaggedfraud": [0, 0, 0, 0],
})

show = create_transaction_features(edge, large_amount_threshold=1000)
show[["type", "amount", "origin_balance_error",
      "destination_balance_error", "destination_balance_inconsistent"]]

## 3. Leakage controls

Two traps this codebase explicitly guards against:

1. **Future leakage** - behavioural features are *expanding* (as-of-now): a
   customer's average at row *t* uses transactions up to *t*, never later ones,
   so a time-based split cannot leak the future.
2. **Test leakage of the amount threshold** - `is_large_transaction` must use a
   threshold fitted on the **training rows** only. `build_and_save_model`
   fits it once and persists it in the model bundle so train/serve match.

Quick check that the first transaction of a customer sees only itself:

In [ ]:
firsts = featured.sort_values(["nameorig", "step"]).groupby("nameorig").head(1)
bad = int((~np.isclose(firsts["customer_avg_amount"], firsts["amount"])).sum())
print("customers whose FIRST transaction has a contaminated average:", bad)
print("(a non-zero value would mean later transactions leaked backwards)")

## 4. The persisted feature sample

`python main.py` exports `featured_transactions.csv` - the same pipeline run on
the first 200k rows of the cleaned file - as a diagnostic artefact.

In [ ]:
header = pd.read_csv(FEATURED_PATH, nrows=0)
print(f"{len(header.columns)} columns in featured_transactions.csv "
      f"({len(header.columns) - len(base.columns)} engineered)")

sample = pd.read_csv(FEATURED_PATH, nrows=100_000)
sample.shape

In [ ]:
# Quick univariate screen: how well does each feature separate the classes
# on its own? (direction-free AUC - an optimistic but useful ranking)
from sklearn.metrics import roc_auc_score

target = sample["isfraud"]
feat_cols = [c for c in sample.columns if c != "isfraud" and sample[c].dtype != object]

rows = []
for c in feat_cols:
    s = pd.to_numeric(sample[c], errors="coerce").fillna(0)
    if target.nunique() < 2 or s.nunique() < 2:
        continue
    auc = roc_auc_score(target, s)
    rows.append({"feature": c, "direction_free_auc": max(auc, 1 - auc)})

assoc = pd.DataFrame(rows).sort_values("direction_free_auc", ascending=False)
assoc = assoc.reset_index(drop=True)
assoc.head(12)

## Summary

- **37 engineered features** from transaction-level rules plus **14**
  per-customer behavioural features (exact counts printed above).
- Balance-consistency errors flag impossible rows without any label - they work
  even on unseen fraud patterns.
- Behavioural features capture *deviation from the customer's own norm*
  (`amount_zscore`, `amount_to_customer_avg_ratio`) and *velocity*
  (`transactions_last_24_steps`), the two classic mule-account signals.
- Leakage is controlled on two fronts: expanding (as-of-now) aggregates and a
  train-fitted amount threshold persisted with the model.
- The next notebook (`04`) feeds these features into the three detectors.